In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# args_schema

# 언어 모델이 호출하는, 도구나 함수가 받는 입력 값에 대한 스키마를 정의
# 스카마에는
# 매개 변수의 이름
# 데이터 타입
# 필수/선택 여부
# 설명
# 제약 조건 (값의 범위, 길이 등)이 해당

# 사용 방법
# 1. Pydantic의 BaseModel 클래스를 상속받는 클래스를 만듬 → 이 클래스가 파라미터 스키마가 됨
from langchain_core.pydantic_v1 import BaseModel, Field
class WeatherInput(BaseModel):
    # 여기에 적는 변수가 하나의 파라미터가 되는 것
    pass

# 아래 셀에서 계속

In [ ]:
# 2. 스키마 클래스에 파라미터 변수를 정의

from langchain_core.pydantic_v1 import BaseModel, Field
class WeatherInput(BaseModel):
    city: str = Field(..., description="날씨를 검색할 도시의 이름 (예: Seoul, New York)")
    temperature_unit: str = Field(default="Celsius", description="온도 단위. 'Celsius' 또는 'Fahrenheit'를 사용합니다.")

In [ ]:
# 3. @tool(args_schema=A)
# - @tool 데코레이터를 사용하여 파이썬 함수를 언어 모델이 호출할 수 있는 도구로 변환
# - args_schema=A의 역할은 "이 도구가 A라는 스키마를 따르는 입력을 전달받는다"를 알려주는 것

@tool(args_schema=WeatherInput)
def get_current_weather(city: str, temperature_unit: str) -> str:
    """주어진 도시의 현재 날씨를 가져옵니다."""
    return f"{city}의 현재 날씨는 {temperature_unit} 기준으로 25도입니다."

# A 스키마를 기반으로 도구 스키마를 만들어서 언어 모델에게 전달
# 도구 스키마가 없다면 언어 모델은
# - 도구 사용법을 알 수 없고
# - 사용자의 질문에서 도구에 전달할 매개 변수에 해당하는 값을 알 수 없음

# args_schema가 없다면 Agent는
# 언어 모델 출력에 대해서 필수 매개 변수 전달 여부, 데이터 타입 일치 여부 등을 확인 X


In [ ]:
# Pydantic의 사용 예 - 중첩된 스키마

from langchain_core.pydantic_v1 import BaseModel, Field

class Address(BaseModel):
    """주소 정보를 담는 스키마"""
    street: str = Field(..., description="도로명 주소")
    city: str = Field(..., description="도시 이름")
    zip_code: str = Field(..., description="우편번호")
# ----------------------    -------------------------------------
class UserProfile(BaseModel):
    """사용자 프로필 스키마"""
    name: str = Field(..., description="사용자 이름")
    address: Address = Field(..., description="사용자의 주소")
    # address 변수는 BaseModel 클래스인 Address의 인스턴스
    
    # address.street, address.city, address.zip_code를 의미
    
    # address.street, address.city, address.zip_code 중 어느 하나라도 
    # Address 클래스에 해당하는 스키마와 일치하지 않으면 Pydantic 객체 생성이 실패

In [ ]:
# Pydantic의 사용 예 - 제약 조건

# 제약 조건은 Field 함수를 사용

# age: int = Field(..., ge=0, le=150): age가 0 이상 150 이하인 정수여야 함을 의미
# query: str = Field(..., min_length=5, max_length=100): query의 길이가 5에서 100 사이여야 함을 의미

In [ ]:
# Pydantic의 사용 예 - Union

from langchain_core.pydantic_v1 import BaseModel, Field

class ItemInput(BaseModel):
    # Union[A, B]는 타입이 A 또는 B
    value: Union[str, int] = Field(..., description="아이템의 값이며 숫자 또는 텍스트가 될 수 있습니다.")

In [ ]:
# Pydantic의 사용 예 - Enum

from enum import Enum

# 열거형 클래스는 미리 정해진 값들을 담는 클래스

# 열거형의 기본 값은 자동으로 할당되는 정수
# 멤버에 값을 명시하지 않으면 자동으로 정수가 할당 (1부터 할당)
# AutoValue.FIRST = 1
# AutoValue.SECOND = 2
# AutoValue.THIRD = 3
class AutoValue(Enum):
    FIRST  
    SECOND
    THIRD  

# 멤버에 값을 명시적으로 할당
class AutoStatus(Enum):
    PENDING = 1
    PROCESSING = 2
    COMPLETED = 3

# 열거형 클래스인데 그 열거형의 값들이 문자열
# 이 경우 타입 str을 적어줘야 함
class Unit(str, Enum):
    CELSIUS = "Celsius"
    FAHRENHEIT = "Fahrenheit"

In [ ]:
# 유효성 검사 - 중첩된 모델 유효성 검사 예제

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field, ValidationError
from langchain.tools import tool

# 1. Pydantic 스키마 정의: 유효성 검사 로직
class UserInfo(BaseModel):
    """주문자 정보를 나타내는 스키마입니다."""
    name: str = Field(..., description="사용자 이름")
    age: int = Field(..., description="사용자 나이", gt=0) # 나이는 0보다 커야 함

class Order(BaseModel):
    """주문 생성을 위한 입력 스키마입니다."""
    user: UserInfo = Field(..., description="주문자 정보")
    # user.name
    # user.age
    item: str = Field(..., description="주문 상품명")


# 2. @tool 데코레이터로 LangChain 도구 생성
# 언어 모델이 { "user": { "name": "김철수", "age": 30 }, "item": "노트북" }를 출력
# AgentExecutor는 
# "user" 키의 값인 { "name": "김철수", "age": 30 }을 UserInfo 객체로 변환
# "item" 키의 값인 "노트북"을 문자열
# 이상 두 개를 사용하여 함수에 전달


@tool(args_schema=Order)
# create_order 함수의 매개 변수가 지켜야 할 규칙에 해당하는 스키마 클래스는 Order
# Order는 매개 변수에 적용하는 규칙이고 UserInfo는 값에 대한 규칙

def create_order(user: UserInfo, item: str) -> str:
    # 스키마 클래스 Order와 일치하게 매개 변수를 사용
    # user는 별도의 
    """주어진 사용자 정보와 상품으로 주문을 생성합니다."""
    return f"{user.name}({user.age}세)의 {item} 주문이 완료되었습니다."


# 3. LLM의 출력을 모방하여 데이터 직접 생성
# tool_calls의 arguments 부분으로 시뮬레이션
# {
#   "tool_calls": [
#     {
#       "function": {
#         "name": "create_order",
#         "arguments": {
#           "user": {
#             "name": "김철수",
#             "age": 30
#           },
#           "item": "노트북"
#         }
#       }
#     }
#   ]
# }

valid_data = {
    "user": {
        "name": "김철수",
        "age": 30
    },
    "item": "노트북"
}

invalid_data = {
    "user": {
        "name": "박영희",
        "age": -5  # 이 부분에서 유효성 검사 실패
    },
    "item": "키보드"
}


# 직접 Pydantic을 이용한 유효성 검사 함수 (LangChain 내부 로직 시뮬레이션)
def validate_llm_output(llm_output: dict):
    """LLM의 출력을 Order 스키마로 검증하는 함수."""
    try:
        # 유효성 검사 목적의 객체 생성
        # **llm_output을 넘기면 BaseModel이 파싱을 해서 각 필드에 대한 유효성을 체크
        validated_order = Order(**llm_output)
        print("유효성 검사 성공!")
        print("검증된 데이터:", validated_order.json(indent=2)) 
        return validated_order
        # llm_output 딕셔너리의 데이터가 Order 스키마의 규칙을 하나라도 위반하면 유효성 검사에 실패
        # 이 경우, Pydantic은 ValidationError를 발생시키며 객체 생성을 중단
    except ValidationError as e:
        print("유효성 검사 실패!")
        print(f"오류: {e}")
        return None


# 5. 실행
print("--- 유효한 데이터 검증 ---")
validate_llm_output(valid_data)

print("\n--- 유효하지 않은 데이터 검증 ---")
validate_llm_output(invalid_data)

In [ ]:
# 유효성 검사 - 열거형 유효성 검사 예제

In [ ]:
from enum import Enum
from langchain_core.pydantic_v1 import BaseModel, Field, ValidationError
from langchain.tools import tool

# 열거형(Enum) 클래스 정의: 허용되는 값들의 집합
class Status(str, Enum):
    PENDING = "대기"
    PROCESSING = "처리 중"
    COMPLETED = "완료"

# Pydantic 스키마
class TaskUpdate(BaseModel):
    """작업 상태 업데이트를 위한 입력 스키마입니다."""
    task_id: int = Field(..., description="업데이트할 작업 ID")
    status: Status = Field(..., description="업데이트할 작업 상태. '대기', '처리 중', '완료' 중 하나여야 합니다.")

@tool(args_schema=TaskUpdate)
def update_task_status(task_id: int, status: Status) -> str:
    """주어진 작업 ID의 상태를 업데이트합니다."""
    return f"작업 ID {task_id}의 상태가 '{status.value}'로 업데이트되었습니다."

valid_data = {
    "task_id": 101,
    "status": "처리 중"
}

invalid_data = {
    "task_id": 102,
    "status": "진행 중"  # Enum에 정의되지 않은 값
}

# 5. 직접 Pydantic을 이용한 유효성 검사 함수 (LangChain 내부 로직 시뮬레이션)
def validate_llm_output(llm_output: dict):
    try:
        validated_task = TaskUpdate(**llm_output)
        print("유효성 검사 성공!")
        print("검증된 데이터:", validated_task.json(indent=2)) 
    except ValidationError as e:
        print("유효성 검사 실패!")
        print(f"오류: {e}")

# --- 실행 ---
print("--- 유효한 데이터 검증 ---")
validate_llm_output(valid_data)

print("\n--- 유효하지 않은 데이터 검증 ---")
validate_llm_output(invalid_data)

In [ ]:
# 유효성 검사 - 제약 조건 유효성 검사

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field, ValidationError
from langchain.tools import tool

class PasswordChange(BaseModel):
    """비밀번호 변경을 위한 입력 스키마입니다."""
    old_password: str = Field(..., description="현재 비밀번호")
    new_password: str = Field(
        ...,
        description="새 비밀번호 (8자 이상 20자 이하)",
        min_length=8,
        max_length=20
    )

@tool(args_schema=PasswordChange)
def change_password(old_password: str, new_password: str) -> str:
    """주어진 비밀번호를 새 비밀번호로 변경합니다."""
    return f"비밀번호가 성공적으로 변경되었습니다."

valid_data = {
    "old_password": "old_password_123",
    "new_password": "new_password_123" # 길이 16, 유효
}

invalid_data = {
    "old_password": "old_password_123",
    "new_password": "short" # 길이 5, min_length 위반
}

def validate_llm_output(llm_output: dict):
    try:
        validated_password = PasswordChange(**llm_output)
        print("유효성 검사 성공!")
        print("검증된 데이터:", validated_password.json(indent=2)) 
    except ValidationError as e:
        print("유효성 검사 실패!")
        print(f"오류: {e}")

print("--- 유효한 데이터 검증 ---")
validate_llm_output(valid_data)

print("\n--- 유효하지 않은 데이터 (길이 위반) 검증 ---")
validate_llm_output(invalid_data)